In [4]:
!apt-get update -y
!apt-get install -y python3-tk xvfb x11vnc websockify novnc

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:4 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Hit:6 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Fetched 3,917 B in 1s (4,230 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
python3-tk is already the newest version (3.12.3-0ubuntu1).
xvfb is already the newest version (2:21.1.12-1ubuntu1.8).
x11vnc 

In [7]:
import json
import ipywidgets as widgets
from IPython.display import display, clear_output

# модель нечіткох бази знань

class Term:
    def __init__(self, name, a, b, c):
        self.name = name
        self.a, self.b, self.c = float(a), float(b), float(c)

    def membership(self, x):
        if x <= self.a or x >= self.c: return 0.0
        if x == self.b: return 1.0
        if x < self.b: return (x - self.a) / (self.b - self.a)
        return (self.c - x) / (self.c - self.b)

class FuzzyVariable:
    def __init__(self, name, minimum, maximum):
        self.name, self.minimum, self.maximum = name, float(minimum), float(maximum)
        self.terms = {}
    def add_term(self, term):
        self.terms[term.name] = term

class FuzzyRule:
    def __init__(self, temp_term, hum_term, vent_term):
        self.temp_term, self.hum_term, self.vent_term = temp_term, hum_term, vent_term

def create_default_db():
    temp = FuzzyVariable("Температура", 0, 40)
    temp.add_term(Term("Низька", 0, 0, 20))
    temp.add_term(Term("Середня", 10, 20, 30))
    temp.add_term(Term("Висока", 20, 40, 40))

    hum = FuzzyVariable("Вологість", 0, 100)
    hum.add_term(Term("Низька", 0, 0, 50))
    hum.add_term(Term("Нормальна", 30, 50, 70))
    hum.add_term(Term("Висока", 50, 100, 100))

    vent = FuzzyVariable("Вентиляція", 0, 100)
    vent.add_term(Term("Низька", 0, 0, 50))
    vent.add_term(Term("Середня", 25, 50, 75))
    vent.add_term(Term("Висока", 50, 100, 100))

    rules = [
        FuzzyRule("Низька", "Низька", "Низька"),
        FuzzyRule("Середня", "Нормальна", "Середня"),
        FuzzyRule("Висока", "Висока", "Висока")
    ]
    return {"Температура": temp, "Вологість": hum, "Вентиляція": vent}, rules

variables, rules = create_default_db()


# інтерфейс в колаб
out = widgets.Output()

var_select = widgets.Dropdown(options=list(variables.keys()), description='Змінна:')
term_select = widgets.Dropdown(description='Терм:')

a_float = widgets.FloatText(description='Параметр a:')
b_float = widgets.FloatText(description='Параметр b:')
c_float = widgets.FloatText(description='Параметр c:')
btn_apply = widgets.Button(description='Застосувати', button_style='primary')

temp_in = widgets.FloatText(value=30, description='Темп, °C:')
hum_in = widgets.FloatText(value=75, description='Вологість, %:')
btn_run = widgets.Button(description='Розрахувати висновок', button_style='success')

def update_terms_options(change):
    var_name = var_select.value
    term_select.options = list(variables[var_name].terms.keys())
    fill_params(None)

def fill_params(change):
    var_name, t_name = var_select.value, term_select.value
    if t_name in variables[var_name].terms:
        term = variables[var_name].terms[t_name]
        a_float.value, b_float.value, c_float.value = term.a, term.b, term.c

var_select.observe(update_terms_options, names='value')
term_select.observe(fill_params, names='value')
update_terms_options(None)

def on_apply_clicked(b):
    var_name, t_name = var_select.value, term_select.value
    term = variables[var_name].terms[t_name]
    term.a, term.b, term.c = a_float.value, b_float.value, c_float.value
    with out:
        clear_output()
        print(f"Оновлено параметри для {var_name} -> {t_name}: a={term.a}, b={term.b}, c={term.c}")

def on_run_clicked(b):
    t_val, h_val = temp_in.value, hum_in.value
    t_m = {k: v.membership(t_val) for k, v in variables["Температура"].terms.items()}
    h_m = {k: v.membership(h_val) for k, v in variables["Вологість"].terms.items()}

    aggregated = {k: 0.0 for k in variables["Вентиляція"].terms}
    for r in rules:
        act = min(t_m.get(r.temp_term, 0.0), h_m.get(r.hum_term, 0.0))
        aggregated[r.vent_term] = max(aggregated[r.vent_term], act)

    num, den = 0.0, 0.0
    for i in range(201):
        x = i * 0.5
        deg = 0.0
        for t_name, act in aggregated.items():
            term = variables["Вентиляція"].terms[t_name]
            deg = max(deg, min(act, term.membership(x)))
        num += x * deg
        den += deg

    res = 0.0 if den == 0 else num / den
    with out:
        clear_output()
        print(f"РЕЗУЛЬТАТ НЕЧІТКОГО ВИСНОВКУ")
        print(f"Розрахований рівень вентиляції: {res:.2f} %")
        print("Ступені активації термів:", aggregated)

btn_apply.on_click(on_apply_clicked)
btn_run.on_click(on_run_clicked)

display(widgets.VBox([
    widgets.HTML("<h3>Редактор нечіткої бази знань (Google Colab)</h3>"),
    widgets.HBox([var_select, term_select]),
    widgets.HBox([a_float, b_float, c_float, btn_apply]),
    widgets.HTML("<h4>Тестування нечіткого висновку</h4>"),
    widgets.HBox([temp_in, hum_in, btn_run]),
    out
]))